# Liu2024 Source-Aligned Short-Scale Riemannian Pilot

**STOPPED ALIGNMENT ROUTE.** Full50 recentering improved only +0.15 points (`p=0.719`). Do not rerun or tune pooling/alignment from these outcomes; see `AGENTS.md` section 2e. Execution fails closed.

# 1. Setup

In [ ]:
raise RuntimeError('CLOSED source-aligned Riemann experiment: see AGENTS.md section 2e.')
from pathlib import Path
from datetime import datetime
import builtins, hashlib, json, os, platform, random, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import linalg, signal
from scipy.special import expit
from scipy.stats import wilcoxon
from sklearn.covariance import OAS
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.distance import distance_riemann
from pyriemann.utils.mean import mean_covariance
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | cwd: {Path.cwd()}")

# 2. Configuration
## 2.1 Locked Domain Defaults

In [ ]:
LIU29 = ["Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4", "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2"]
LIU29_RAW_INDICES = list(range(17)) + list(range(18, 30))
MOTOR13 = ["F3", "F4", "FCz", "FC3", "FC4", "Cz", "C3", "C4", "CP3", "CP4", "Pz", "P3", "P4"]
MOTOR13_RAW_INDICES = [LIU29_RAW_INDICES[LIU29.index(name)] for name in MOTOR13]
EXPECTED_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 51)]
PRESPECIFIED_TARGETS = ["sub-01", "sub-03", "sub-07"]
LOCKED_METHODS = ["target_only", "pooled_unaligned", "riemannian_recentered"]
assert len(MOTOR13_RAW_INDICES) == 13

## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-aligned-short-scale-riemann-pilot"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "locked_shallow_artifact": str(WORKING_DIR / "artifacts" / "liu2024-compact-mi-models" / "20260712_165746_790145_fd8ab986"),
    "covariance_cache_dir": str(WORKING_DIR / "artifacts" / "covariance_cache" / "source_aligned_short_scale_riemann"),
    "experiment_name": "source_aligned_short_scale_riemann_pilot",
    "config_note": "Prespecified 01/03/07 pilot using 49 labeled source subjects and locked exactly-once target OOF folds.",

    # ------------------------------------------------------------------
    # Dataset / immutable provenance
    # ------------------------------------------------------------------
    "target_subjects": ["sub-01", "sub-03", "sub-07"],
    "expected_subject_count": 50,
    "expected_trials_per_subject": 40,
    "expected_source_subject_count": 49,
    "expected_shallow_run_id": "20260712_165746_790145_fd8ab986",
    "expected_global_split_hash": "801ec1d2c981335f",
    "marker_channel_index": 32,
    "onset_marker_value": 2,
    "onset_plausible_range": [800, 1300],
    "onset_fallback_sample": 1003,
    "native_sfreq": 500,
    "mi_window_s": [0.0, 4.0],

    # ------------------------------------------------------------------
    # Fixed preprocessing and covariance views
    # ------------------------------------------------------------------
    "spatial_mode": "motor13",
    "filter_context": "full_trial_then_crop",
    "average_reference": True,
    "bands_hz": [[8.0, 12.0], [13.0, 20.0], [20.0, 30.0], [8.0, 30.0]],
    "temporal_scales": {"1s": {"length_s": 1.0, "starts_s": [0.0, 1.0, 2.0, 3.0]}, "2s": {"length_s": 2.0, "starts_s": [0.0, 1.0, 2.0]}},
    "filter_order": 4,
    "covariance_estimator": "oas",
    "trace_normalize": True,

    # ------------------------------------------------------------------
    # Model / evaluation
    # ------------------------------------------------------------------
    "methods": ["target_only", "pooled_unaligned", "riemannian_recentered"],
    "tangent_metric": "riemann",
    "lda_solver": "lsqr",
    "lda_shrinkage": "auto",
    "margin_scale_eps": 1e-8,
    "evaluation_mode": "reuse_locked_stratified_5fold",
    "cv_folds": 5,
    "bootstrap_iterations": 10000,
    "bootstrap_seed": 202607,

    # ------------------------------------------------------------------
    # Reproducibility / diagnostics
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "collapse_threshold": 0.9,
    "cache_policy": "reuse_or_create",
    "run_synthetic_checks": False,
}


In [ ]:
TARGETS = list(CONFIG["target_subjects"])
if TARGETS not in (PRESPECIFIED_TARGETS, EXPECTED_SUBJECTS) or CONFIG["methods"] != LOCKED_METHODS:
    raise ValueError("Targets must be the prespecified smoke set or frozen full50 expansion; methods are immutable")
locked = (CONFIG["spatial_mode"] == "motor13" and CONFIG["filter_context"] == "full_trial_then_crop" and CONFIG["average_reference"] is True and CONFIG["covariance_estimator"] == "oas" and CONFIG["trace_normalize"] is True)
if not locked or CONFIG["bands_hz"] != [[8.0, 12.0], [13.0, 20.0], [20.0, 30.0], [8.0, 30.0]]:
    raise ValueError("Preprocessing and bands must match the locked same-fold fusion protocol")
if CONFIG["temporal_scales"] != {"1s": {"length_s": 1.0, "starts_s": [0.0, 1.0, 2.0, 3.0]}, "2s": {"length_s": 2.0, "starts_s": [0.0, 1.0, 2.0]}}:
    raise ValueError("Temporal scales must match the locked same-fold fusion protocol")
print(json.dumps({k: CONFIG[k] for k in ["experiment_name", "target_subjects", "methods", "expected_global_split_hash", "bands_hz", "temporal_scales"]}, indent=2))

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try:
        stream.write(text)
    except UnicodeEncodeError:
        encoding = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(encoding, errors="replace").decode(encoding, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep, end = kwargs.pop("sep", " "), kwargs.pop("end", "\n")
    flush, target = kwargs.pop("flush", False), kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ""
    for stream in ([target] if target is not None else [sys.stdout, _LOG_FILE_HANDLE]):
        _safe_write_text(stream, stamped + end)
    if flush:
        _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2, allow_nan=False)
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Locked Split and Source Assertions

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
def stable_hash(payload, length=16):
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:length]
def load_json(path):
    with open(path, encoding="utf-8") as stream:
        return json.load(stream)
def validate_locked_splits():
    root = Path(CONFIG["locked_shallow_artifact"]).resolve()
    required = ["config.json", "run_metadata.json", "subject_inventory.csv", "splits.json"]
    missing = [name for name in required if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(f"Locked artifact missing: {missing}")
    metadata, splits = load_json(root / "run_metadata.json"), load_json(root / "splits.json")
    inventory = pd.read_csv(root / "subject_inventory.csv")
    if metadata.get("run_id") != CONFIG["expected_shallow_run_id"] or list(splits) != EXPECTED_SUBJECTS:
        raise AssertionError("Locked artifact identity or subject order mismatch")
    if inventory["subject_id"].tolist() != EXPECTED_SUBJECTS or set(inventory["split_hash"].astype(str)) != {CONFIG["expected_global_split_hash"]}:
        raise AssertionError("Locked inventory split mismatch")
    lookup, assertions = {}, []
    for sid in TARGETS:
        if stable_hash(splits[sid]) != CONFIG["expected_global_split_hash"]:
            raise AssertionError(f"Split hash mismatch for {sid}")
        seen = []
        for split in splits[sid]:
            fold_id = int(split["fold_id"])
            train = np.asarray(split["train_indices"], dtype=int)
            test = np.asarray(split["test_indices"], dtype=int)
            if len(train) != 32 or len(test) != 8 or len(np.intersect1d(train, test)) or sorted(np.r_[train, test].tolist()) != list(range(40)):
                raise AssertionError(f"Invalid locked partition for {sid} fold {fold_id}")
            lookup[(sid, fold_id)] = {"train_indices": train, "test_indices": test}
            seen.extend(test.tolist())
            assertions.append({"target_subject": sid, "fold_id": fold_id, "split_hash": CONFIG["expected_global_split_hash"], "train_count": 32, "test_count": 8, "disjoint": True})
        if sorted(seen) != list(range(40)) or len(set(seen)) != 40:
            raise AssertionError(f"Non-exact OOF coverage in locked splits for {sid}")
    provenance = {"locked_artifact": str(root), "run_id": metadata["run_id"], "global_split_hash": CONFIG["expected_global_split_hash"], "file_sha256": {name: sha256_file(root / name) for name in required}}
    return lookup, assertions, provenance
LOCKED_SPLITS, SPLIT_ASSERTIONS, SPLIT_PROVENANCE = validate_locked_splits()
SOURCE_IDS = {target: [sid for sid in EXPECTED_SUBJECTS if sid != target] for target in TARGETS}
for target, sources in SOURCE_IDS.items():
    if target in sources or len(sources) != 49 or len(set(sources)) != 49:
        raise AssertionError(f"Target/source exclusion failed for {target}")

## 3.2 Independent Full-Trial Covariance Extraction

In [ ]:
VIEW_SPECS = [{"id": f"{scale}_{start:g}s_{lo:g}-{hi:g}Hz", "scale": scale, "start_s": float(start), "length_s": float(spec["length_s"]), "band_hz": [float(lo), float(hi)]} for scale, spec in CONFIG["temporal_scales"].items() for start in spec["starts_s"] for lo, hi in CONFIG["bands_hz"]]
assert len(VIEW_SPECS) == 28 and sum(v["scale"] == "1s" for v in VIEW_SPECS) == 16 and sum(v["scale"] == "2s" for v in VIEW_SPECS) == 12
def covariance_signature():
    keys = ["native_sfreq", "mi_window_s", "marker_channel_index", "onset_marker_value", "onset_plausible_range", "onset_fallback_sample", "spatial_mode", "filter_context", "average_reference", "bands_hz", "temporal_scales", "filter_order", "covariance_estimator", "trace_normalize"]
    payload = {key: CONFIG[key] for key in keys}
    payload.update({"channel_names": MOTOR13, "raw_indices": MOTOR13_RAW_INDICES, "view_specs": VIEW_SPECS})
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:16]
def selected_files():
    files = sorted(Path(CONFIG["source_extract_dir"]).glob("sub-*/sub-*_task-motor-imagery_eeg.mat"))
    if [path.parent.name for path in files] != EXPECTED_SUBJECTS:
        raise AssertionError("Exactly the ordered full 50 raw files are required")
    return files
def load_subject(path):
    sid = path.parent.name
    eeg = sio.loadmat(path)["eeg"][0, 0]
    raw = np.asarray(eeg["rawdata"], dtype=np.float64)
    labels = np.asarray(eeg["label"]).reshape(-1).astype(int)
    if set(np.unique(labels)).issubset({1, 2}):
        labels -= 1
    if raw.shape[0] != 40 or labels.shape != (40,) or np.bincount(labels, minlength=2).tolist() != [20, 20]:
        raise ValueError(f"Unexpected trial/label structure for {sid}")
    marker = raw[:, CONFIG["marker_channel_index"], :]
    lo, hi = CONFIG["onset_plausible_range"]
    detected = []
    for row in marker:
        hits = np.flatnonzero(row == CONFIG["onset_marker_value"])
        valid = hits[(hits >= lo) & (hits <= hi)]
        detected.append(int(valid[0]) if len(valid) else -1)
    plausible = [value for value in detected if lo <= value <= hi]
    fallback = int(np.median(plausible)) if plausible else int(CONFIG["onset_fallback_sample"])
    onsets = np.asarray([value if lo <= value <= hi else fallback for value in detected], dtype=int)
    return sid, raw[:, MOTOR13_RAW_INDICES, :].copy(), labels, onsets, {"source_path": str(path.resolve()), "source_sha256": sha256_file(path), "marker_fallback_count": int(np.sum(np.asarray(detected) < 0))}
def estimate_oas(window):
    fitted = OAS(store_precision=False, assume_centered=True).fit(window.T)
    covariance = fitted.covariance_ / np.trace(fitted.covariance_)
    eigenvalues = np.linalg.eigvalsh(covariance)
    if not np.all(np.isfinite(covariance)) or eigenvalues[0] <= 0 or not np.isclose(np.trace(covariance), 1.0):
        raise ValueError("Invalid OAS trace-normalized SPD covariance")
    probabilities = eigenvalues / eigenvalues.sum()
    return covariance, [float(fitted.shrinkage_), float(np.exp(-np.sum(probabilities * np.log(probabilities)))), float(eigenvalues[-1] / eigenvalues[0])]
def extract_covariances(trials, onsets):
    covariances = np.empty((40, 28, 13, 13), dtype=np.float64)
    diagnostics = np.empty((40, 28, 3), dtype=np.float64)
    filters = {tuple(band): signal.butter(CONFIG["filter_order"], band, btype="bandpass", fs=CONFIG["native_sfreq"], output="sos") for band in CONFIG["bands_hz"]}
    for trial_i, trial in enumerate(trials):
        referenced = trial - trial.mean(axis=0, keepdims=True)
        filtered = {band: signal.sosfiltfilt(sos, referenced, axis=-1) for band, sos in filters.items()}
        for view_i, view in enumerate(VIEW_SPECS):
            start = int(onsets[trial_i] + round(view["start_s"] * CONFIG["native_sfreq"]))
            stop = start + int(round(view["length_s"] * CONFIG["native_sfreq"]))
            window = filtered[tuple(view["band_hz"])][:, start:stop]
            if window.shape != (13, int(round(view["length_s"] * CONFIG["native_sfreq"]))):
                raise ValueError(f"Incomplete marker-relative view {view['id']}")
            covariances[trial_i, view_i], diagnostics[trial_i, view_i] = estimate_oas(window)
    return covariances, diagnostics
def get_covariances(path):
    sid, trials, labels, onsets, provenance = load_subject(path)
    cache_root = Path(CONFIG["covariance_cache_dir"]) / covariance_signature()
    cache_root.mkdir(parents=True, exist_ok=True)
    cache_path = cache_root / f"{sid}.npz"
    manifest = {"signature": covariance_signature(), "subject_id": sid, "source_sha256": provenance["source_sha256"], "labels": labels.tolist(), "onsets": onsets.tolist(), "view_ids": [v["id"] for v in VIEW_SPECS]}
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        valid = set(cached.files) == {"manifest_json", "covariances", "diagnostics"} and json.loads(str(cached["manifest_json"].item())) == manifest and cached["covariances"].shape == (40, 28, 13, 13) and cached["diagnostics"].shape == (40, 28, 3) and np.all(np.isfinite(cached["covariances"]))
        if not valid:
            raise ValueError(f"Incompatible covariance cache refused: {cache_path}")
        covariances, diagnostics, cache_status = cached["covariances"], cached["diagnostics"], "reused"
    else:
        covariances, diagnostics = extract_covariances(trials, onsets)
        np.savez_compressed(cache_path, manifest_json=np.asarray(json.dumps(manifest, sort_keys=True)), covariances=covariances, diagnostics=diagnostics)
        cache_status = "created"
    provenance.update({"subject_id": sid, "cache_path": str(cache_path), "cache_status": cache_status, "covariance_signature": covariance_signature()})
    return sid, covariances, labels, diagnostics, provenance

# 4. Model
## 4.1 Fold-Local Source Recentring and Tangent Shrinkage-LDA

In [ ]:
def spd_power(matrix, power):
    values, vectors = linalg.eigh(matrix, check_finite=True)
    if values[0] <= 0:
        raise ValueError("SPD power received a non-SPD matrix")
    return (vectors * (values ** power)) @ vectors.T
def transport_covariances(covariances, source_center, target_center):
    transform = spd_power(target_center, 0.5) @ spd_power(source_center, -0.5)
    transported = np.asarray([transform @ covariance @ transform.T for covariance in covariances])
    if not np.all(np.linalg.eigvalsh(transported) > 0):
        raise ValueError("Transport produced a non-SPD covariance")
    return transported
def oriented_score(model, features):
    scores = np.asarray(model.decision_function(features), dtype=float).reshape(-1)
    if model.classes_.tolist() == [0, 1]:
        return scores
    if model.classes_.tolist() == [1, 0]:
        return -scores
    raise ValueError("Expected binary classes 0/1")
def margin_scale(scores):
    scale = float(np.median(np.abs(scores)))
    if not np.isfinite(scale) or scale < CONFIG["margin_scale_eps"]:
        scale = float(np.sqrt(np.mean(np.asarray(scores) ** 2)))
    if not np.isfinite(scale) or scale < CONFIG["margin_scale_eps"]:
        raise ValueError("Degenerate training margins")
    return scale
def fit_view(train_covariances, train_labels, test_covariances):
    tangent = TangentSpace(metric=CONFIG["tangent_metric"]).fit(train_covariances)
    train_features = tangent.transform(train_covariances)
    test_features = tangent.transform(test_covariances)
    scaler = StandardScaler().fit(train_features)
    train_features, test_features = scaler.transform(train_features), scaler.transform(test_features)
    model = LinearDiscriminantAnalysis(solver=CONFIG["lda_solver"], shrinkage=CONFIG["lda_shrinkage"]).fit(train_features, train_labels)
    scale = margin_scale(oriented_score(model, train_features))
    return oriented_score(model, test_features) / scale, scale
def precompute_source_centers(all_data):
    centers = {}
    for sid in EXPECTED_SUBJECTS:
        centers[sid] = np.asarray([mean_covariance(all_data[sid]["covariances"][:, view_i], metric="riemann") for view_i in range(len(VIEW_SPECS))])
    return centers

## 4.2 Synthetic Mathematical Checks

In [ ]:
def run_synthetic_checks():
    rng = np.random.default_rng(202607)
    def random_spd():
        matrix = rng.normal(size=(5, 5))
        return matrix @ matrix.T + np.eye(5)
    source_center, target_center = random_spd(), random_spd()
    transported_center = transport_covariances(source_center[None], source_center, target_center)[0]
    if not np.allclose(transported_center, target_center, rtol=1e-9, atol=1e-9):
        raise AssertionError("Congruence transport does not map source center to target center")
    source_covariances = np.asarray([random_spd() for _ in range(12)])
    transported = transport_covariances(source_covariances, source_center, target_center)
    before = np.asarray([distance_riemann(covariance, source_center) for covariance in source_covariances])
    after = np.asarray([distance_riemann(covariance, target_center) for covariance in transported])
    if not np.allclose(before, after, rtol=1e-8, atol=1e-8):
        raise AssertionError("Affine-invariant distances were not preserved")
    labels = np.repeat([0, 1], 10)
    train_covariances = np.asarray([random_spd() for _ in labels])
    scores, scale = fit_view(train_covariances, labels, train_covariances[:4])
    probabilities = expit(scores)
    if scores.shape != (4,) or not np.all(np.isfinite(probabilities)) or scale <= 0:
        raise AssertionError("Synthetic tangent-LDA check failed")
    return {"center_mapping": True, "affine_distance_invariance": True, "transported_spd": True, "finite_margin_normalized_scores": True}
if CONFIG["run_synthetic_checks"]:
    print(run_synthetic_checks())

# 5. Training
## 5.1 Leakage-Safe Outer-Fold Runner

In [ ]:
def fold_record(target, fold_id, method, train_indices, test_indices, labels, scores):
    probability_1 = expit(scores)
    probabilities = np.column_stack([1.0 - probability_1, probability_1])
    predictions = (scores >= 0).astype(int)
    majority = float(max(np.mean(predictions == 0), np.mean(predictions == 1)))
    return {"subject_id": target, "fold_id": int(fold_id), "method": method, "train_indices": train_indices.tolist(), "test_indices": test_indices.tolist(), "true_labels": labels.tolist(), "scores": scores.tolist(), "probabilities": probabilities.tolist(), "predictions": predictions.tolist(), "accuracy": float(accuracy_score(labels, predictions)), "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)), "confusion_matrix": confusion_matrix(labels, predictions, labels=[0, 1]).tolist(), "prediction_histogram": np.bincount(predictions, minlength=2).tolist(), "collapse_diagnostics": {"majority_prediction_fraction": majority, "collapsed": bool(majority >= CONFIG["collapse_threshold"])}, "outer_test_used_for_center": False, "outer_test_used_for_fit": False, "outer_test_used_for_selection": False}
def run_fold(target, fold_id, all_data, source_centers):
    split = LOCKED_SPLITS[(target, fold_id)]
    train_indices, test_indices = split["train_indices"], split["test_indices"]
    if len(np.intersect1d(train_indices, test_indices)):
        raise AssertionError("Target train/test overlap")
    target_data = all_data[target]
    sources = SOURCE_IDS[target]
    if target in sources or len(sources) != CONFIG["expected_source_subject_count"]:
        raise AssertionError("Target present in source pool")
    source_labels = np.concatenate([all_data[sid]["labels"] for sid in sources])
    target_train_labels = target_data["labels"][train_indices]
    method_view_scores = {method: [] for method in LOCKED_METHODS}
    margin_rows, center_rows = [], []
    for view_i, view in enumerate(VIEW_SPECS):
        target_train = target_data["covariances"][train_indices, view_i]
        target_test = target_data["covariances"][test_indices, view_i]
        source_unaligned = np.concatenate([all_data[sid]["covariances"][:, view_i] for sid in sources])
        target_center = mean_covariance(target_train, metric="riemann")
        source_aligned = np.concatenate([transport_covariances(all_data[sid]["covariances"][:, view_i], source_centers[sid][view_i], target_center) for sid in sources])
        training_sets = {
            "target_only": (target_train, target_train_labels),
            "pooled_unaligned": (np.concatenate([source_unaligned, target_train]), np.concatenate([source_labels, target_train_labels])),
            "riemannian_recentered": (np.concatenate([source_aligned, target_train]), np.concatenate([source_labels, target_train_labels])),
        }
        for method, (train_covariances, train_labels) in training_sets.items():
            scores, fitted_scale = fit_view(train_covariances, train_labels, target_test)
            method_view_scores[method].append(scores)
            margin_rows.append({"target_subject": target, "fold_id": fold_id, "method": method, "view_id": view["id"], "n_fit_trials": len(train_labels), "training_margin_scale": float(fitted_scale), "fit_scope": "method_training_only"})
        target_center_hash = hashlib.sha256(np.round(target_center, 12).tobytes()).hexdigest()
        for sid in sources:
            mapped_center = transport_covariances(source_centers[sid][view_i][None], source_centers[sid][view_i], target_center)[0]
            center_rows.append({"target_subject": target, "fold_id": fold_id, "source_subject": sid, "view_id": view["id"], "source_center_trial_count": 40, "source_center_scope": "source_subject_only", "target_center_trial_count": 32, "target_center_indices": json.dumps(train_indices.tolist()), "target_test_indices_excluded": bool(not np.isin(test_indices, train_indices).any()), "target_center_sha256": target_center_hash, "source_center_sha256": hashlib.sha256(np.round(source_centers[sid][view_i], 12).tobytes()).hexdigest(), "center_distance_before_transport": float(distance_riemann(source_centers[sid][view_i], target_center)), "transported_center_target_distance": float(distance_riemann(mapped_center, target_center))})
    results, prediction_rows = [], []
    for method, score_list in method_view_scores.items():
        score_matrix = np.column_stack(score_list)
        scale_scores = [score_matrix[:, [i for i, spec in enumerate(VIEW_SPECS) if spec["scale"] == scale]].mean(axis=1) for scale in ["1s", "2s"]]
        scores = np.mean(np.column_stack(scale_scores), axis=1)
        result = fold_record(target, fold_id, method, train_indices, test_indices, target_data["labels"][test_indices], scores)
        results.append(result)
        for row_i, trial_i in enumerate(test_indices):
            prediction_rows.append({"subject_id": target, "fold_id": fold_id, "trial_index": int(trial_i), "method": method, "true_label": int(target_data["labels"][trial_i]), "prediction": int(result["predictions"][row_i]), "score": float(scores[row_i]), "probability_0": float(result["probabilities"][row_i][0]), "probability_1": float(result["probabilities"][row_i][1])})
    return results, prediction_rows, margin_rows, center_rows

## 5.2 Run Prespecified Targets

In [ ]:
print("=" * 72)
print(json.dumps(CONFIG, indent=2, sort_keys=True))
print("=" * 72)
ALL_DATA, INVENTORY_ROWS, COVARIANCE_DIAGNOSTIC_ROWS = {}, [], []
for path in selected_files():
    sid, covariances, labels, diagnostics, provenance = get_covariances(path)
    ALL_DATA[sid] = {"covariances": covariances, "labels": labels}
    INVENTORY_ROWS.append({**provenance, "n_trials": 40, "class_0": 20, "class_1": 20})
    for view_i, view in enumerate(VIEW_SPECS):
        COVARIANCE_DIAGNOSTIC_ROWS.append({"subject_id": sid, "view_id": view["id"], "mean_oas_shrinkage": float(diagnostics[:, view_i, 0].mean()), "mean_effective_rank": float(diagnostics[:, view_i, 1].mean()), "mean_condition_number": float(diagnostics[:, view_i, 2].mean()), "all_trace_one": bool(np.allclose(np.trace(covariances[:, view_i], axis1=1, axis2=2), 1.0))})
if list(ALL_DATA) != EXPECTED_SUBJECTS:
    raise AssertionError("Full50 covariance inventory mismatch")
SOURCE_CENTERS = precompute_source_centers(ALL_DATA)
FOLD_RESULTS, PREDICTION_ROWS, MARGIN_ROWS, CENTER_ASSERTIONS = [], [], [], []
for target in TARGETS:
    for fold_id in range(CONFIG["cv_folds"]):
        fold_results, prediction_rows, margin_rows, center_rows = run_fold(target, fold_id, ALL_DATA, SOURCE_CENTERS)
        FOLD_RESULTS.extend(fold_results); PREDICTION_ROWS.extend(prediction_rows); MARGIN_ROWS.extend(margin_rows); CENTER_ASSERTIONS.extend(center_rows)
    print(f"{target}: completed all 5 locked folds with 49 source subjects")
PREDICTIONS = pd.DataFrame(PREDICTION_ROWS).sort_values(["subject_id", "method", "trial_index"]).reset_index(drop=True)
coverage = PREDICTIONS.groupby(["subject_id", "method"])["trial_index"].agg(["count", "nunique"])
if len(FOLD_RESULTS) != len(TARGETS) * CONFIG["cv_folds"] * len(LOCKED_METHODS) or len(PREDICTIONS) != len(TARGETS) * 40 * len(LOCKED_METHODS) or not coverage.eq(40).all().all():
    raise AssertionError("Exactly-once OOF output cardinality failed")

# 6. Results
## 6.1 Subject, Global, and Paired Metrics

In [ ]:
def bootstrap_mean_ci(values):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(CONFIG["bootstrap_seed"])
    draws = rng.choice(values, size=(CONFIG["bootstrap_iterations"], len(values)), replace=True).mean(axis=1)
    return [float(x) for x in np.percentile(draws, [2.5, 97.5])]
subject_rows = []
for (sid, method), rows in PREDICTIONS.groupby(["subject_id", "method"], sort=True):
    rows = rows.sort_values("trial_index")
    if rows["trial_index"].tolist() != list(range(40)):
        raise AssertionError(f"Incomplete pooled OOF predictions for {sid} {method}")
    subject_rows.append({"subject_id": sid, "method": method, "n_trials": 40, "accuracy": float(accuracy_score(rows.true_label, rows.prediction)), "balanced_accuracy": float(balanced_accuracy_score(rows.true_label, rows.prediction)), "confusion_matrix": confusion_matrix(rows.true_label, rows.prediction, labels=[0, 1]).tolist()})
SUBJECT_METRICS = pd.DataFrame(subject_rows)
GLOBAL_METRICS = {"primary_estimand": "mean_target_subject_pooled_exactly_once_oof_balanced_accuracy", "n_target_subjects": len(TARGETS), "target_subjects": TARGETS, "source_subjects_per_target": 49, "global_split_hash": CONFIG["expected_global_split_hash"], "methods": {}}
for method in LOCKED_METHODS:
    values = SUBJECT_METRICS.loc[SUBJECT_METRICS.method == method, "balanced_accuracy"].to_numpy()
    GLOBAL_METRICS["methods"][method] = {"mean_subject_balanced_accuracy": float(values.mean()), "subject_bootstrap_95_ci": bootstrap_mean_ci(values), "n_subjects": len(values)}
pivot = SUBJECT_METRICS.pivot(index="subject_id", columns="method", values="balanced_accuracy").loc[TARGETS]
PAIRED_DELTAS = {}
for method in ["pooled_unaligned", "riemannian_recentered"]:
    deltas = (pivot[method] - pivot["target_only"]).to_numpy()
    statistic,p_value=(None,None) if len(TARGETS)==3 else wilcoxon(deltas,alternative="two-sided",zero_method="wilcox")
    PAIRED_DELTAS[method] = {"reference": "target_only", "subject_deltas": {sid: float(pivot.loc[sid, method] - pivot.loc[sid, "target_only"]) for sid in TARGETS}, "mean_delta": float(deltas.mean()), "mean_delta_points": float(100 * deltas.mean()), "paired_subject_bootstrap_95_ci": bootstrap_mean_ci(deltas), "wins": int(np.sum(deltas > 0)), "ties": int(np.sum(deltas == 0)), "losses": int(np.sum(deltas < 0)), "n_subjects": len(TARGETS), "wilcoxon_statistic": None if statistic is None else float(statistic), "wilcoxon_p_value": None if p_value is None else float(p_value), "reason": "bounded three-target pilot" if len(TARGETS)==3 else "frozen exploratory full50 expansion"}
GLOBAL_METRICS["paired_vs_target_only"] = PAIRED_DELTAS
GLOBAL_METRICS["protocol"] = {"view_selection": False, "hyperparameter_selection": False, "source_labels_used": True, "target_excluded_from_source": True, "source_centers": "each source subject's 40 covariances, separately per view", "target_centers": "32 target outer-training covariances, separately per view", "target_test_used_for_centers_or_fit": False, "score_normalization": "method-training-margin-only", "view_fusion": "equal within scale, then equal across fixed 1s/2s scales"}
print(json.dumps(GLOBAL_METRICS, indent=2))

## 6.2 Performance Visualizations

In [ ]:
colors = {"target_only": "#355070", "pooled_unaligned": "#b56576", "riemannian_recentered": "#e56b6f"}
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(TARGETS)); width = 0.24
for offset, method in enumerate(LOCKED_METHODS):
    ax.bar(x + (offset - 1) * width, 100 * pivot[method], width, label=method.replace("_", " "), color=colors[method])
ax.axhline(50, color="black", linestyle="--", linewidth=1); ax.set_xticks(x[::max(1,len(TARGETS)//25)], [sid.replace("sub-", "") for sid in TARGETS][::max(1,len(TARGETS)//25)]); ax.set(xlabel="Target subject", ylabel="Pooled OOF balanced accuracy (%)", title="Source-Aligned Short-Scale Riemann Pilot"); ax.legend(); fig.tight_layout()
subject_plot_path = ARTIFACT_DIR / "source_aligned_subject_performance.png"; fig.savefig(subject_plot_path, dpi=180); plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
means = [100 * GLOBAL_METRICS["methods"][method]["mean_subject_balanced_accuracy"] for method in LOCKED_METHODS]
ax.bar(range(3), means, color=[colors[m] for m in LOCKED_METHODS]); ax.axhline(50, color="black", linestyle="--"); ax.set_xticks(range(3), [m.replace("_", " ") for m in LOCKED_METHODS], rotation=15, ha="right"); ax.set(ylabel="Mean target BA (%)", title="Bounded Pilot Method Comparison"); fig.tight_layout()
global_plot_path = ARTIFACT_DIR / "source_aligned_global_performance.png"; fig.savefig(global_plot_path, dpi=180); plt.close(fig)
rows = PREDICTIONS[PREDICTIONS.method == "riemannian_recentered"]; matrix = confusion_matrix(rows.true_label, rows.prediction, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4)); image = ax.imshow(matrix, cmap="Blues")
for i in range(2):
    for j in range(2): ax.text(j, i, str(matrix[i, j]), ha="center", va="center")
ax.set(xticks=[0, 1], yticks=[0, 1], xlabel="Predicted", ylabel="True", title="Recentered Aggregated OOF Confusion"); fig.colorbar(image, ax=ax); fig.tight_layout()
confusion_plot_path = ARTIFACT_DIR / "source_aligned_aggregated_confusion.png"; fig.savefig(confusion_plot_path, dpi=180); plt.close(fig)

## 6.3 Experiment Summary

In [ ]:
for method in LOCKED_METHODS:
    print(f"{method}: {100 * GLOBAL_METRICS['methods'][method]['mean_subject_balanced_accuracy']:.2f}% mean target BA")
for method, comparison in PAIRED_DELTAS.items():
    print(f"{method} vs target_only: {comparison['mean_delta_points']:+.2f} points")

## 6.4 Fail-Closed Leakage Assertions

In [ ]:
LEAKAGE_ASSERTIONS = {
    "configured_targets_exact": list(PREDICTIONS.subject_id.drop_duplicates()) == TARGETS,
    "methods_exact": set(PREDICTIONS.method) == set(LOCKED_METHODS) and PREDICTIONS.method.nunique() == len(LOCKED_METHODS),
    "split_hash_exact": SPLIT_PROVENANCE["global_split_hash"] == CONFIG["expected_global_split_hash"],
    "exactly_once_oof": bool(coverage.eq(40).all().all()),
    "all_targets_excluded_from_sources": all(target not in SOURCE_IDS[target] and len(SOURCE_IDS[target]) == 49 for target in TARGETS),
    "all_target_centers_use_32_training_trials": all(row["target_center_trial_count"] == 32 for row in CENTER_ASSERTIONS),
    "all_source_centers_source_subject_only": all(row["source_center_trial_count"] == 40 and row["source_center_scope"] == "source_subject_only" for row in CENTER_ASSERTIONS),
    "all_source_centers_map_to_target_center": all(row["transported_center_target_distance"] < 1e-6 for row in CENTER_ASSERTIONS),
    "all_target_test_indices_excluded_from_centers": all(row["target_test_indices_excluded"] is True for row in CENTER_ASSERTIONS),
    "outer_test_used_for_center": False,
    "outer_test_used_for_fit": False,
    "outer_test_used_for_selection": False,
    "hyperparameter_selection_present": False,
}
required_true = [key for key in LEAKAGE_ASSERTIONS if key not in {"outer_test_used_for_center", "outer_test_used_for_fit", "outer_test_used_for_selection", "hyperparameter_selection_present"}]
required_false = ["outer_test_used_for_center", "outer_test_used_for_fit", "outer_test_used_for_selection", "hyperparameter_selection_present"]
if not all(LEAKAGE_ASSERTIONS[key] is True for key in required_true) or not all(LEAKAGE_ASSERTIONS[key] is False for key in required_false):
    raise AssertionError(f"Leakage assertion failure: {LEAKAGE_ASSERTIONS}")
print(json.dumps(LEAKAGE_ASSERTIONS, indent=2))

## 6.5 Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"; json.dump(FOLD_RESULTS, open(cv_results_path, "w"), indent=2, allow_nan=False)
fold_metrics_path = ARTIFACT_DIR / "fold_metrics.csv"; pd.DataFrame([{k: v for k, v in row.items() if k not in {"train_indices", "test_indices", "true_labels", "scores", "probabilities", "predictions", "confusion_matrix", "prediction_histogram", "collapse_diagnostics"}} for row in FOLD_RESULTS]).to_csv(fold_metrics_path, index=False)
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"; json.dump(subject_rows, open(subject_metrics_path, "w"), indent=2, allow_nan=False)
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"; json.dump(GLOBAL_METRICS, open(global_metrics_path, "w"), indent=2, allow_nan=False)
predictions_path = ARTIFACT_DIR / "predictions.csv"; PREDICTIONS.to_csv(predictions_path, index=False)
paired_deltas_path = ARTIFACT_DIR / "paired_deltas_vs_target_only.json"; json.dump(PAIRED_DELTAS, open(paired_deltas_path, "w"), indent=2, allow_nan=False)
split_assertions_path = ARTIFACT_DIR / "split_assertions.json"; json.dump({"provenance": SPLIT_PROVENANCE, "folds": SPLIT_ASSERTIONS, "leakage_assertions": LEAKAGE_ASSERTIONS}, open(split_assertions_path, "w"), indent=2, allow_nan=False)
source_assertions_path = ARTIFACT_DIR / "source_assertions.json"; json.dump({target: {"source_subjects": SOURCE_IDS[target], "source_count": len(SOURCE_IDS[target]), "target_excluded": target not in SOURCE_IDS[target]} for target in TARGETS}, open(source_assertions_path, "w"), indent=2, allow_nan=False)
center_assertions_path = ARTIFACT_DIR / "center_assertions.csv"; pd.DataFrame(CENTER_ASSERTIONS).to_csv(center_assertions_path, index=False)
margin_scales_path = ARTIFACT_DIR / "training_margin_scales.csv"; pd.DataFrame(MARGIN_ROWS).to_csv(margin_scales_path, index=False)
covariance_diagnostics_path = ARTIFACT_DIR / "covariance_diagnostics.csv"; pd.DataFrame(COVARIANCE_DIAGNOSTIC_ROWS).to_csv(covariance_diagnostics_path, index=False)
provenance_path = ARTIFACT_DIR / "data_provenance.json"; json.dump(INVENTORY_ROWS, open(provenance_path, "w"), indent=2, allow_nan=False)
artifact_paths = {"config": str(config_path), "run_log": str(LOG_PATH), "cv_results": str(cv_results_path), "fold_metrics": str(fold_metrics_path), "subject_metrics": str(subject_metrics_path), "global_metrics": str(global_metrics_path), "predictions": str(predictions_path), "paired_deltas": str(paired_deltas_path), "split_assertions": str(split_assertions_path), "source_assertions": str(source_assertions_path), "center_assertions": str(center_assertions_path), "training_margin_scales": str(margin_scales_path), "covariance_diagnostics": str(covariance_diagnostics_path), "data_provenance": str(provenance_path), "subject_plot": str(subject_plot_path), "global_plot": str(global_plot_path), "confusion_plot": str(confusion_plot_path)}
run_metadata = {"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"], "target_subjects": TARGETS, "source_subjects_per_target": 49, "n_channels": 13, "channel_names": MOTOR13, "view_specs": VIEW_SPECS, "methods": LOCKED_METHODS, "split_hash": CONFIG["expected_global_split_hash"], "covariance_signature": covariance_signature(), "seed": BASE_SEED, "leakage_assertions": LEAKAGE_ASSERTIONS, "global_metrics": GLOBAL_METRICS, "artifacts": artifact_paths}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"; json.dump(run_metadata, open(run_metadata_path, "w"), indent=2, allow_nan=False)
print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass